# Módulo 8 — Taller integrador: de los datos a una decisión

**Curso: Análisis y Pronóstico de Datos Mineros con Python**

> Recorrer el flujo completo sobre un caso minero y terminar en una recomendación respaldada por evidencia.

---

### Cómo usar este notebook
1. Ábrelo en **Google Colab** y ejecuta las celdas **de arriba hacia abajo**.
2. Los datos se descargan solos desde el repositorio del curso; no tienes que subir nada.
3. Lee las salidas: cada bloque responde una pregunta concreta, no ejecutes por ejecutar.

In [ ]:
# Librerías base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # gráficos estadísticos (preinstalado en Colab)

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.width', 120)

# --- Datos del curso -------------------------------------------------
# Los CSV viven en el repositorio del curso y se descargan solos.
# Si no hubiera internet, la función pide subir el archivo a mano.
REPO_DATOS = 'https://raw.githubusercontent.com/HishanFarfan/curso-datos-mineros/main/datos'

def cargar_datos(nombre, **kw):
    try:
        return pd.read_csv(f'{REPO_DATOS}/{nombre}', **kw)
    except Exception as e:
        print('No se pudo descargar desde GitHub:', e)
    try:
        from google.colab import files          # Colab: subir a mano
        print(f"Sube '{nombre}':")
        return pd.read_csv(next(iter(files.upload())), **kw)
    except ModuleNotFoundError:
        return pd.read_csv(nombre, **kw)         # local: archivo en el cwd

## Bloque 0 — El problema

> Una operación dispone de registros horarios de una planta concentradora (`datos_proceso_planta.csv`). Se requiere **caracterizar** el comportamiento de la recuperación de cobre, **evaluar** si puede pronosticarse de forma útil y **detectar** episodios en que el proceso se alejó de lo esperado.

Completa antes de programar:

- Variable analizada: __________
- Unidad: __________
- Frecuencia temporal: __________
- Periodo cubierto: __________
- Pregunta operacional concreta: __________

## Bloque 1 — Carga y preparación (M1)

In [ ]:
df = cargar_datos('datos_proceso_planta.csv', sep=';', decimal=',', encoding='latin-1')
df.columns = df.columns.str.strip()
df['Tonelaje_tph'] = df['Tonelaje_tph'].replace(-999, np.nan)
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df = (df.sort_values('Fecha')
        .groupby('Fecha', as_index=False).first()
        .set_index('Fecha').asfreq('h'))
df.isna().sum()

In [ ]:
# TODO: decide e implementa el tratamiento de faltantes. Documenta el porqué.
serie = df['Recuperacion_pct']
serie = serie.interpolate('time', limit=2, limit_area='inside')
serie = serie.dropna()

## Bloque 2 — Exploración estadística (M2)

In [ ]:
# TODO: describe la serie; compara por Turno y Tipo_mineral; matriz de correlación.
serie.describe()

## Bloque 3 — Estructura temporal (M3)

In [ ]:
# TODO: serie, media/std móviles, descomposición STL (period=24).
serie.plot(); plt.show()

## Bloque 4 — Diagnóstico (M4)

In [ ]:
# TODO: ACF/PACF, ADF/KPSS en nivel y diferenciado. Decide transformación, d, D, m.
from statsmodels.tsa.stattools import adfuller
print('ADF nivel p =', round(adfuller(serie)[1], 4))
print('ADF Δ     p =', round(adfuller(serie.diff().dropna())[1], 4))

## Bloque 5 — Modelamiento (M5)

In [ ]:
# TODO: 2-3 candidatos ARIMA/SARIMA, tabla AIC/BIC, residuos + Ljung-Box.
from statsmodels.tsa.statespace.sarimax import SARIMAX

## Bloque 6 — Pronóstico y validación (M6)

In [ ]:
# TODO: split cronológico, forecast + intervalo, MAE/RMSE/MAPE, baseline naïve, backtesting.
n_train = int(len(serie) * 0.8)
train, test = serie.iloc[:n_train], serie.iloc[n_train:]

## Bloque 7 — Anomalías (M7)

In [ ]:
# TODO: residuos del modelo -> umbral / intervalo. ¿Eventos aislados o cambio persistente?
# Recuerda el cambio de campaña de mineral A->B.

## Bloque 8 — Conclusión operacional

Redacta un informe de **una página** con esta estructura:

| Sección | Contenido |
|---|---|
| **Objetivo** | Qué se quería responder |
| **Datos** | Periodo, frecuencia, limpieza aplicada y por qué |
| **Método** | Diagnóstico, modelos comparados, validación |
| **Resultados** | MAE/RMSE, mejora sobre naïve, autocorrelación residual, anomalías |
| **Recomendación** | Uso propuesto del modelo, horizonte útil, límites, qué revisar |

Cada decisión debe seguir la lógica **evidencia → decisión**. Ejemplo: *«la ACF mostró picos en múltiplos de 24 → se incorporó componente estacional»*.

### Errores a evitar
modelar sin mirar los datos · borrar outliers automáticamente · train/test aleatorio · elegir modelo solo por AIC · reportar solo métricas · confundir anomalía con causa · entregar forecast sin incertidumbre · recomendar algo que los datos no respaldan.

---
## Cierre

El taller empieza con una base de datos y termina con una **recomendación respaldada por evidencia**. Ese es el arco completo del curso: datos → información → modelo → evidencia → decisión.